In [ ]:
!pip install -q huggingface_hub
!pip install -q jiwer
!pip install -q librosa torchaudio torchcodec
("IMPORTANT INSTALLATION DONE")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.9 MB/s eta 0:00:00


'IMPORTANT INSTALLATION DONE'

In [ ]:
import torchcodec
import datasets
import librosa
import numpy as np
from datasets import load_dataset
from datasets import Audio
import os
import torch
import torchaudio
import random
import torch.nn as nn
import librosa
from jiwer import wer, cer
import numpy as np
import pandas as pd

In [ ]:
dataset = load_dataset("Purvaxxx/hindi_test_dataset")
print(dataset)

DatasetDict({
    hindi: Dataset({
        features: ['chunked_audio_filepath', 'text', 'en_text'],
        num_rows: 100
    })
})


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
def preprocess_audio(audio_dict, target_sr=16000, n_fft=1024, hop_length=256, n_mels=80, normalize="zscore"):
    audio = audio_dict["array"]
    sr = audio_dict["sampling_rate"]

    # Resample if needed
    if sr != target_sr:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
        sr = target_sr

    # Normalize waveform (peak normalization)
    if np.max(np.abs(audio)) > 0:
        audio = audio / np.max(np.abs(audio))

    # Ensure minimum length
    if len(audio) < n_fft:
        audio = np.pad(audio, (0, n_fft - len(audio)), mode="constant")

    # Mel spectrogram
    melspec = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=n_mels, n_fft=n_fft, hop_length=hop_length
    )

    # Convert to log scale
    log_melspec = librosa.power_to_db(melspec, ref=np.max).astype(np.float32)

    # Normalization
    if normalize == "zscore":
        mean = np.mean(log_melspec)
        std = np.std(log_melspec) + 1e-9
        log_melspec = (log_melspec - mean) / std
    elif normalize == "minmax":
        min_val = np.min(log_melspec)
        max_val = np.max(log_melspec)
        log_melspec = (log_melspec - min_val) / (max_val - min_val + 1e-9)

    # Return (time, n_mels)
    return log_melspec.T


In [ ]:
hindi_chars = [
    ' ', 'अ','आ','इ','ई','उ','ऊ','ऋ','ए','ऐ','ओ','औ','अं','अः',
    'क','ख','ग','घ','ङ','च','छ','ज','झ','ञ','ट','ठ','ड','ढ','ण',
    'त','थ','द','ध','न','प','फ','ब','भ','म','य','र','ल','व','श','ष','स','ह',
    'ा','ि','ी','ु','ू','े','ै','ो','ौ','ं','ः','्','।', ',', '.'
]

vocab = ["[PAD]", "[SOS]", "[EOS]"] + hindi_chars

# Build mapping tables
char_to_id = {ch: i for i, ch in enumerate(vocab)}
id_to_char = {i: ch for i, ch in enumerate(vocab)}

def encode(text):
    # Ignore characters not in vocab
    return [char_to_id["[SOS]"]] + [char_to_id[ch] for ch in text if ch in char_to_id] + [char_to_id["[EOS]"]]

def decode(ids):
    return "".join(
        [id_to_char[i] for i in ids if i not in (char_to_id["[SOS]"], char_to_id["[EOS]"], char_to_id["[PAD]"])]
    )

print(vocab)
print(len(vocab))


['[PAD]', '[SOS]', '[EOS]', ' ', 'अ', 'आ', 'इ', 'ई', 'उ', 'ऊ', 'ऋ', 'ए', 'ऐ', 'ओ', 'औ', 'अं', 'अः', 'क', 'ख', 'ग', 'घ', 'ङ', 'च', 'छ', 'ज', 'झ', 'ञ', 'ट', 'ठ', 'ड', 'ढ', 'ण', 'त', 'थ', 'द', 'ध', 'न', 'प', 'फ', 'ब', 'भ', 'म', 'य', 'र', 'ल', 'व', 'श', 'ष', 'स', 'ह', 'ा', 'ि', 'ी', 'ु', 'ू', 'े', 'ै', 'ो', 'ौ', 'ं', 'ः', '्', '।', ',', '.']
65


In [ ]:
# POSITIONAL ENCODING
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=1000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

print("positional encoding done")

positional encoding done


In [ ]:
# TRANSFORMER ASR MODEL
class TransformerASR(nn.Module):
    def __init__(self,
                 input_dim,
                 d_model,
                 nhead,
                 num_encoder_layers,
                 num_decoder_layers,
                 dim_feedforward,
                 dropout,
                 vocab_size,
                 max_encoder_len,
                 max_decoder_len,
                ):
        super().__init__()
        self.input_linear = nn.Linear(input_dim, d_model)
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_encoder_len, dropout=dropout)
        self.pos_decoder = PositionalEncoding(d_model, max_len=max_decoder_len, dropout=dropout)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )

        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self,
                src,
                tgt,
                src_key_padding_mask=None,
                tgt_key_padding_mask=None,
                memory_key_padding_mask=None,
                tgt_mask=None
               ):
        src = self.input_linear(src)
        src = self.pos_encoder(src)

        tgt = self.embedding(tgt)
        tgt = self.pos_decoder(tgt)

        output = self.transformer(
            src, tgt,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=memory_key_padding_mask,
            tgt_mask=tgt_mask
        )

        output = self.fc_out(output)
        return output

    def generate(self, src, start_token_id, end_token_id, max_len=100):
        self.eval()
        with torch.no_grad():
            src = self.input_linear(src)
            src = self.pos_encoder(src)
            memory = self.transformer.encoder(src)

            batch_size = src.size(0)
            ys = torch.full((batch_size, 1), start_token_id, dtype=torch.long, device=src.device)

            for _ in range(max_len - 1):
                tgt = self.embedding(ys)
                tgt = self.pos_decoder(tgt)

                tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(src.device)
                output = self.transformer.decoder(tgt, memory, tgt_mask=tgt_mask)
                output = self.fc_out(output[:, -1, :])
                next_token = torch.argmax(output, dim=1, keepdim=True)
                ys = torch.cat([ys, next_token], dim=1)

                if torch.all(next_token.squeeze() == end_token_id):
                    break

        return ys

print("ARCHITECTURE DEFINED")

ARCHITECTURE DEFINED


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TransformerASR(
    input_dim=80,
    d_model=256,
    nhead=4,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dim_feedforward=512,
    dropout=0.1,
    vocab_size = len(vocab),
    max_encoder_len=17362,
    max_decoder_len=2898
).to(device)

In [ ]:
Transformer_ASR = "/content/drive/MyDrive/VOICE2ENGLISH/transformer_asr_finetuned.pt"
asr_checkpoint = torch.load(Transformer_ASR, map_location=device)
model.load_state_dict(asr_checkpoint["model_state_dict"])
model.to(device)
model.eval()

TransformerASR(
  (input_linear): Linear(in_features=80, out_features=256, bias=True)
  (embedding): Embedding(65, 256)
  (pos_encoder): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (pos_decoder): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-2): 3 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
          )
          (linear1): Linear(in_features=256, out_features=512, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=512, out_features=256, bias=True)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropou

In [ ]:
preprocessed_test = []

test_dataset_hindi = dataset["hindi"]

for i in range(len(test_dataset_hindi)):
    sample = test_dataset_hindi[i]

    audio_dict = sample["chunked_audio_filepath"]
    mel = preprocess_audio(audio_dict)

    preprocessed_test.append({
        "log_melspec": mel,
        "text": sample["text"]
    })

In [ ]:
SOS_ID = char_to_id["[SOS]"]
EOS_ID = char_to_id["[EOS]"]

pred_texts = []

for i, sample in enumerate(preprocessed_test[:100]):
    mel_np = sample["log_melspec"]
    true_text = sample["text"]

    mel = torch.from_numpy(mel_np).float().unsqueeze(0).to(device)
    pred_ids = model.generate(mel, start_token_id=SOS_ID, end_token_id=EOS_ID, max_len=150)[0].tolist()

    pred_text = decode(pred_ids)
    pred_texts.append(pred_text)

    if i < 10:
        print(f"True: {true_text}")
        print(f"Predicted: {pred_text}")
        print("******")


True: पर फरीसियों और व्यवस्थापकों ने उस से बपतिस्मा न लेकर परमेश्वर की मनसा को अपने विषय में टाल दिया
Predicted: वरन फरीसियों और व्यवस्थापु को ने उससे बपतिस्मा न लेकर परमेश्वर की मंशा को अपने विषय में टाल दिया
******
True: सो मैं इस युग के लोगों की उपमा किस से दूं कि वे किस के समान हैं
Predicted: सो मैं इस युग के लोगों की उपमा किस से दूं कि वे किसके समान हैं
******
True: वे उन बालकों के समान हैं जो बाजार में बैठे हुए एक दूसरे से पुकारकर कहते हैं कि हम ने तुम्हारे लिये बांसली बजाई और तुम न नाचे हम ने विलाप किया और तुम न रोए
Predicted: फिर उन बालकों के समान हैं जो बजार में बैठे हुए एक दूसरे से पुकारकर कहते हैं हमने तुम्हारे लिए बांस योजाई और तुम न न चे हमने विलाप किया और तुम नरोए
******
True: क्योंकि यूहन्ना बपतिस्मा देने वाला न रोटी खाता आया न दाखरस पीता आया और तुम कहते हो उस में दुष्टात्मा है
Predicted: क्योंकि योहन्ना बक्तिस्मा देने वाला नरोटी खाता आया न दाखरस पिता आया और तुम कहते हो उसमें दुष्टात्मा है
******
True: मनुष्य का पुत्र खाता पीता आया है और तुम कहते हो देखो पेटू और पियक्कड़

In [ ]:
true_texts = [sample["text"] for sample in preprocessed_test[:100]]

worderror = wer(true_texts, pred_texts)

print(f"WER on test set : {worderror * 100:.2f}%")


WER on test set : 31.43%


In [ ]:
from IPython.display import Audio as IPyAudio, display
import librosa

audio_sample1 = "/content/drive/MyDrive/VOICE2ENGLISH/hindi_sample_1.wav"
array, sr = librosa.load(audio_sample1, sr=16000)

display(IPyAudio(array, rate=sr))
audio_dict = {
    "array": array,
    "sampling_rate": sr
}

mel = preprocess_audio(audio_dict)
print("Shape of log-mel spectrogram:", mel.shape)

mel = torch.from_numpy(mel).float().unsqueeze(0).to(device)
pred_ids = model.generate(mel, start_token_id=SOS_ID, end_token_id=EOS_ID, max_len=150)[0].tolist()
pred_text = decode(pred_ids)

print(f"Predicted : {pred_text}")

Shape of log-mel spectrogram: (41, 80)
Predicted : और कहने लगा


In [ ]:
audio_sample2 = "/content/drive/MyDrive/VOICE2ENGLISH/hindi_sample_17.wav"
array, sr = librosa.load(audio_sample2, sr=16000)

display(IPyAudio(array, rate=sr))
audio_dict = {
    "array": array,
    "sampling_rate": sr
}
mel = preprocess_audio(audio_dict)
print("Shape of log-mel spectrogram:", mel.shape)

mel = torch.from_numpy(mel).float().unsqueeze(0).to(device)
pred_ids = model.generate(mel, start_token_id=SOS_ID, end_token_id=EOS_ID, max_len=150)[0].tolist()
pred_text = decode(pred_ids)

print(f"Predicted : {pred_text}")

Shape of log-mel spectrogram: (88, 80)
Predicted : मैं उठकर अपने घर चला गया
